# Hotel Voice Agent - Workshop Walkthrough

This notebook walks through building a LiveKit voice agent for hotel booking:
1. **State A**: LiveKit voice agent with STT/LLM/TTS pipeline and tools
2. **State B**: Simulation with FutureAGI Simulate SDK (LiveKit engine)
3. **Evaluation**: Voice-specific evals (conciseness, conversation quality)
4. **Optimization**: GEPA evolutionary prompt optimization

**Note**: The voice agent runs as a standalone process. This notebook covers the architecture,
simulation, evaluation, and optimization parts.

In [ ]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

## State A: Voice Agent Architecture

The voice agent (in `agent.py`) uses this pipeline:
```
Caller Speech -> STT (Deepgram) -> LLM (Groq/OpenAI) -> TTS (OpenAI) -> Spoken Response
                                     |
                              Tool Calls (check rooms, book, etc.)
```

Let's look at the tools and the (intentionally flawed) system prompt.

In [ ]:
from hotel_voice.tools import ROOMS, HOTEL_AMENITIES

print("Room Types:")
for key, room in ROOMS.items():
    print(f"  {room['type']}: ${room['price_per_night']}/night (capacity: {room['capacity']})")

print(f"\nHotel Amenities: {len(HOTEL_AMENITIES)}")
for a in HOTEL_AMENITIES[:5]:
    print(f"  - {a}")
print("  ...")

In [ ]:
# The flawed system prompt - notice the markdown formatting!
SYSTEM_PROMPT = """You are a hotel receptionist at The Grand Hotel. Help guests with:
- Room availability and pricing
- Making reservations
- Cancellations and modifications
- Hotel amenities information

Use the available tools to look up information. Be professional and helpful.
Always provide complete details including:
* Room type options
* Pricing breakdowns
* Available dates
* Amenity lists"""

print("Current system prompt:")
print(SYSTEM_PROMPT)
print("\n--- Problems for voice: bullet points, markdown, verbose 'complete details' ---")

## Running the Voice Agent

To test the voice agent live:
```bash
python hotel_voice/agent.py dev
```
Then connect via LiveKit Playground or a WebRTC client.

## State B: Simulate with FutureAGI

The Simulate SDK's LiveKit engine connects a simulated caller to your deployed voice agent.

In [ ]:
import os
from fi.simulate import AgentDefinition, SimulatorAgentDefinition, TestRunner, Scenario, Persona

LIVEKIT_URL = os.environ.get("LIVEKIT_URL", "wss://localhost:7880")

agent_def = AgentDefinition(
    name="Hotel Receptionist",
    url=LIVEKIT_URL,
    room_name="hotel-notebook-test",
    system_prompt=SYSTEM_PROMPT,
)

simulator = SimulatorAgentDefinition(
    name="sim-caller",
    instructions="You are a realistic hotel caller. Keep responses conversational.",
    llm={"model": "gpt-4o-mini"},
    tts={"model": "tts-1", "voice": "echo"},
    allow_interruptions=True,
)

scenario = Scenario(
    name="hotel-notebook",
    dataset=[
        Persona(
            persona={"name": "David", "mood": "efficient"},
            situation="Needs a suite for Thursday-Sunday.",
            outcome="Quick booking, concise responses.",
        ),
        Persona(
            persona={"name": "Ryan", "mood": "price-sensitive"},
            situation="Wants cheapest room for two nights.",
            outcome="Clear price quote, no overwhelming details.",
        ),
    ],
)

print("Simulation configured.")
print("NOTE: Run 'python hotel_voice/agent.py dev' first!")

In [ ]:
# Run simulation (requires voice agent running)
runner = TestRunner()
report = await runner.run_test(
    agent_definition=agent_def,
    scenario=scenario,
    simulator=simulator,
    record_audio=True,
)

for r in report.results:
    print(f"\n--- {r.persona.persona['name']} ---")
    print(r.transcript[:500])
    if r.audio_combined_path:
        print(f"Audio: {r.audio_combined_path}")

## Evaluation

In [ ]:
from fi.simulate import evaluate_report

report = evaluate_report(
    report,
    eval_templates=["task_completion", "is_concise", "tone", "conversation_quality"],
    model_name="turing_flash",
)

for r in report.results:
    name = r.persona.persona["name"]
    print(f"\n--- {name} ---")
    if r.evaluation:
        for tmpl, scores in r.evaluation.items():
            print(f"  {tmpl}: {scores.get('score', 'N/A')} - {scores.get('reason', '')[:150]}")

## Optimization with GEPA

GEPA uses **evolutionary optimization** with Pareto-aware selection.
It's the most powerful optimizer - ideal for balancing competing objectives:
- Conciseness (short for voice)
- Warmth (friendly receptionist)
- Accuracy (correct info from tools)
- Voice-friendliness (no markdown, natural numbers)

In [ ]:
from config import get_litellm_model
from fi.opt.generators import LiteLLMGenerator
from fi.opt.optimizers import GEPAOptimizer
from fi.opt.base.evaluator import Evaluator
from fi.opt.datamappers import BasicDataMapper
from fi.evals.metrics import CustomLLMJudge
from fi.evals.llm import LiteLLMProvider

model = get_litellm_model()

judge = CustomLLMJudge(
    provider=LiteLLMProvider(),
    config={"name": "voice_judge", "grading_criteria": (
        "Score 0-1 for a VOICE hotel receptionist: voice-friendliness (no markdown, "
        "natural numbers), conciseness, warmth, task completion."
    )},
    model=model,
)

evaluator = Evaluator(metric=judge)
mapper = BasicDataMapper(key_map={"response": "generated_output", "expected_response": "answer"})

dataset = [
    {"guest_message": "What's your cheapest room?", "answer": "Our Standard Room is a hundred twenty a night with a queen bed. Would you like to book one?"},
    {"guest_message": "Do you have a pool?", "answer": "Yes, outdoor pool open seven A.M. to ten P.M. We also have a spa and fitness center."},
    {"guest_message": "I need a suite Thursday to Sunday", "answer": "A suite is three-fifty a night. Let me check Thursday to Sunday. Could I get your name?"},
]

initial_prompt = SYSTEM_PROMPT + "\n\nGuest says: {guest_message}"

optimizer = GEPAOptimizer(reflection_model=model, generator_model=model)
result = optimizer.optimize(
    evaluator=evaluator, data_mapper=mapper, dataset=dataset,
    initial_prompts=[initial_prompt], max_metric_calls=100,
)

print(f"Score: {result.final_score:.4f}")
print(f"\nOptimized prompt:\n{result.best_generator.get_prompt_template()}")